# 2-тарау. Автоматты дифференциалдау

*«Машина қалай үйренеді? 2-том» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 2.1. Градиентті кім есептейді?

## 2.2. requires_grad: амалдарды жазып ал

In [ ]:
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x + 1
y.backward()          # dy/dx есептеу
print(y)
print(x.grad)         # 2x + 2 = 8

## 2.3. Есептеу графы

In [ ]:
def show(fn, depth=0):
    print("  " * depth + type(fn).__name__)
    for nxt, _ in fn.next_functions:
        if nxt is not None:
            show(nxt, depth + 1)

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x + 1
show(y.grad_fn)

In [ ]:
def f(x):
    if x > 0:
        return x ** 3      # оң сандар үшін
    return -x              # теріс сандар үшін

x = torch.tensor(2.0, requires_grad=True)
f(x).backward()
print(x.grad)              # 3x² = 12

## 2.4. 1-томдағы есепті тексереміз

In [ ]:
x, y = torch.tensor(2.0), torch.tensor(1.0)
w = torch.tensor(0.5, requires_grad=True)
b = torch.tensor(-0.5, requires_grad=True)

a = torch.sigmoid(w * x + b)
L = (a - y) ** 2
L.backward()
print(f"a = {a.item():.4f},  L = {L.item():.4f}")
print(f"dL/dw = {w.grad.item():.4f},  dL/db = {b.grad.item():.4f}")

## 2.5. Градиенттер жинақталады

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
for step in range(3):
    loss = (3 * w - 6) ** 2
    loss.backward()
    print(step, w.grad.item())

## 2.6. Градиенттік түсу autograd арқылы

In [ ]:
torch.manual_seed(42)
X = torch.rand(100, 1) * 10
y = 3 * X + 2 + torch.randn(100, 1)    # шын заңдылық: y = 3x + 2

w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.02                              # оқыту қарқыны
for epoch in range(2001):
    pred = X * w + b                   # 1) тура есептеу
    loss = ((pred - y) ** 2).mean()    # 2) қате (MSE)
    loss.backward()                    # 3) градиенттер
    with torch.no_grad():              # 4) жаңарту графқа жазылмайды
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_()                     # 5) градиенттерді нөлдеу
    b.grad.zero_()
    if epoch % 400 == 0:
        print(f"{epoch:4d}: шығын={loss.item():7.3f}  "
              f"w={w.item():.3f}  b={b.item():.3f}")

## 2.7. no_grad және detach

In [ ]:
with torch.no_grad():
    p = X[:3] * w + b
print(p.requires_grad)
z = (X * w).detach()
print(z.requires_grad)
print(round(w.item(), 2))   # .item() — тензордан Python саны